In [9]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [10]:
# 1. Load Dataset
df = pd.read_csv("income.csv")
df.columns = df.columns.str.strip()

# Remove missing values if any
df = df.replace(["?", ""], np.nan).dropna().reset_index(drop=True)

In [11]:
# 2. Separate columns
cat_cols = ['sex', 'education', 'marital-status',
            'workclass', 'occupation']

cont_cols = ['age', 'education-num', 'hours-per-week']

label_col = 'income'

# Remove duplicate label column
df = df[cat_cols + cont_cols + [label_col]]

# Encode categorical columns
encoders = {}

for col in cat_cols:
    encoders[col] = LabelEncoder()
    df[col] = encoders[col].fit_transform(df[col].astype(str))

# Encode target
label_encoder = LabelEncoder()
df[label_col] = label_encoder.fit_transform(df[label_col])

In [12]:
# 3. Create arrays and tensors
X_cat = df[cat_cols].values
X_cont = df[cont_cols].values.astype(np.float32)
y = df[label_col].values

X_cat = torch.tensor(X_cat, dtype=torch.long)
X_cont = torch.tensor(X_cont, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)


In [13]:
# 4. Split dataset
if len(df) < 30000:
    raise ValueError("Dataset must contain 30000 valid rows.")

Xc_train, Xc_test, Xn_train, Xn_test, y_train, y_test = train_test_split(
    X_cat, X_cont, y,
    train_size=25000,
    test_size=5000,
    random_state=42,
    stratify=y
)

print("Training records:", len(y_train))
print("Testing records:", len(y_test))

Training records: 25000
Testing records: 5000


In [14]:
# 5. Define TabularModel
class TabularModel(nn.Module):

    def __init__(self, cat_sizes, num_cont, output_size):
        super().__init__()

        self.embeddings = nn.ModuleList([
            nn.Embedding(size, min(50, (size + 1) // 2))
            for size in cat_sizes
        ])

        self.bn = nn.BatchNorm1d(num_cont)

        total = sum(e.embedding_dim for e in self.embeddings) + num_cont

        self.fc1 = nn.Linear(total, 50)
        self.dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(50, output_size)

    def forward(self, x_cat, x_cont):

        x = [emb(x_cat[:, i])
             for i, emb in enumerate(self.embeddings)]

        x.append(self.bn(x_cont))

        x = torch.cat(x, dim=1)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x


In [15]:
# 6. Set random seed
torch.manual_seed(42)
np.random.seed(42)

# Create model
cat_sizes = [len(encoders[col].classes_) for col in cat_cols]

model = TabularModel(
    cat_sizes,
    len(cont_cols),
    len(label_encoder.classes_)
)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# DataLoader
train_dataset = TensorDataset(Xc_train, Xn_train, y_train)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

In [17]:
# 7. Train model for 300 epochs
for epoch in range(300):

    model.train()
    total_loss = 0

    for xc, xn, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(xc, xn)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 50 == 0:
        print("Epoch:", epoch + 1,
              "Loss:", round(total_loss / len(train_loader), 4))


Epoch: 50 Loss: 0.2362
Epoch: 100 Loss: 0.2363
Epoch: 150 Loss: 0.2368
Epoch: 200 Loss: 0.2358
Epoch: 250 Loss: 0.2339
Epoch: 300 Loss: 0.2357


In [18]:
# 8. Evaluate model
model.eval()

with torch.no_grad():

    outputs = model(Xc_test, Xn_test)

    test_loss = criterion(outputs, y_test).item()

    predictions = torch.argmax(outputs, dim=1)

    accuracy = (predictions == y_test).float().mean().item()

print("\nTest Results")
print("Test Loss:", round(test_loss, 4))
print("Test Accuracy:", round(accuracy * 100, 2), "%")


Test Results
Test Loss: 0.2501
Test Accuracy: 88.52 %


In [23]:
# 9. Bonus: Predict new income
def predict_income(new_data):

    cat_values = []

    for col in cat_cols:
        value = str(new_data[col]).strip()

        if value not in encoders[col].classes_:
            raise ValueError("Unknown category in " + col)

        cat_values.append(encoders[col].transform([value])[0])

    xc = torch.tensor([cat_values], dtype=torch.long)

    xn = torch.tensor(
        [[float(new_data[col]) for col in cont_cols]],
        dtype=torch.float32
    )

    model.eval()

    with torch.no_grad():
        output = model(xc, xn)
        result = torch.argmax(output, dim=1).item()

    return label_encoder.inverse_transform([result])[0]
print("\nProgram executed successfully!")



Program executed successfully!
